# Paper 34 backbone on YOLO26n: Wave (Haar wavelet enhancement) + Cw2PSA

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 34:** **Wave** bypass after the C3k2 of each backbone stage (layers 2, 4, 6, 8) and/or **Cw2PSA** instead of the C2PSA at layer 10 |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged |
| Head (Detect, layer 23) | YOLO26n base, unchanged (the extra stride-4 small-target head of Paper 34 is not used) |

Three variants are trained:

| Run | Backbone |
|---|---|
| `bb_wave` | Wave after the C3k2 at layers 2, 4, 6, 8 |
| `bb_cw2psa` | Cw2PSA instead of the C2PSA at layer 10 |
| `bb_wave_cw2psa` | both together (the Paper 34 backbone) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

**Split run:** this notebook trains **only `bb_cw2psa`** (one of three notebooks, one per variant, so each run fits in one 12 h session). Its output files are named `paper34_bb_cw2psa_...`, so the three results do not overwrite each other.

## 1. Setup

In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 4.7 MB/s eta 0:00:00


In [2]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so wave_backbone.py can be imported

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128
GPU: Tesla T4


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [3]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 11426.73it/s]


train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


## 3. Paper 34 backbone: Wave and Cw2PSA

**Wave module** (Paper 34, Eq. 1–8):
1. Fixed **Haar 2-D DWT** as a grouped 2×2 stride-2 conv (groups = C) → 4 sub-bands LL, LH, HL, HH (H/2 × W/2).
2. Each sub-band is upsampled back to H × W (bilinear) and the 4 are concatenated (4C channels).
3. 1×1 conv (no bias) + BN → C channels.
4. Residual: output = input + result.

**Cw2PSA** (Paper 34, Eq. 10–13): the C2PSA layout with the *identity* half replaced by a Wave branch:
`cv1` → split → branch 1: **Wave**, branch 2: **PSABlock** (unchanged) → concat → `cv2`.

**Where they sit in YOLO26n.**

| Layer | Base YOLO26n | `bb_wave` | `bb_cw2psa` | `bb_wave_cw2psa` | Output |
|---|---|---|---|---|---|
| 2 | C3k2 | C3k2 → **Wave** | C3k2 | C3k2 → **Wave** | 160 × 160 |
| 4 | C3k2 | C3k2 → **Wave** | C3k2 | C3k2 → **Wave** | 80 × 80 |
| 6 | C3k2 | C3k2 → **Wave** | C3k2 | C3k2 → **Wave** | 40 × 40 |
| 8 | C3k2 | C3k2 → **Wave** | C3k2 | C3k2 → **Wave** | 20 × 20 |
| 10 | C2PSA | C2PSA | **Cw2PSA** | **Cw2PSA** | 20 × 20 |

Wave is placed *inside* layers 2, 4, 6, 8 (the C3k2 output goes through Wave), so no layer index changes and the neck
connections stay the same.

**Pretrained weights (exact warm start).** `yolo26n.pt` is loaded first, then the layers are changed:
- every C3k2 and the whole C2PSA (`cv1`, PSA block, `cv2`) keep their pretrained weights;
- the Haar kernels are fixed (not trained); the BN scale (γ) after each Wave 1×1 conv starts at **0**, so every Wave
  branch adds nothing at the start and the model gives exactly the base output. Only the Wave 1×1 conv + BN are learned.

**Not taken from Paper 34:** the extra stride-4 (160 × 160) small-target head, because it is a head change and is costly on
the Raspberry Pi 5.

**Assumptions** (not stated exactly in Paper 34): Wave is placed after the C3k2 of each backbone stage (layers 2, 4, 6, 8),
and the BN γ starts at 0 (Paper 34 does not describe the initialisation).

In [4]:
%%writefile wave_backbone.py
"""Paper 34 backbone for YOLO26n: Wave (fixed Haar DWT, bilinear up, 1x1 + BN, residual) after the C3k2 at layers 2, 4, 6, 8
and/or Cw2PSA (C2PSA with the identity half replaced by a Wave branch) at layer 10."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C2PSA, C3k2
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

WAVE_LAYERS = (2, 4, 6, 8)
CW2PSA_LAYER = 10


def haar_kernels():
    """The 4 Haar 2x2 kernels LL, LH, HL, HH (outer products of hL = [1, 1]/sqrt2 and hH = [1, -1]/sqrt2)."""
    hl = torch.tensor([1.0, 1.0]) / math.sqrt(2)
    hh = torch.tensor([1.0, -1.0]) / math.sqrt(2)
    return torch.stack([torch.outer(hl, hl), torch.outer(hl, hh), torch.outer(hh, hl), torch.outer(hh, hh)])  # (4, 2, 2)


class Wave(nn.Module):
    """Haar DWT (fixed grouped conv, stride 2) -> bilinear upsample x2 -> concat (4C) -> 1x1 conv + BN -> residual add."""

    def __init__(self, c):
        super().__init__()
        self.c = c
        self.register_buffer("haar", haar_kernels().unsqueeze(1).repeat(c, 1, 1, 1))  # (4C, 1, 2, 2), fixed
        self.fuse = nn.Conv2d(4 * c, c, 1, bias=False)
        self.bn = nn.BatchNorm2d(c)
        nn.init.zeros_(self.bn.weight)  # the branch adds nothing at the start (exact warm start)

    def forward(self, x):
        s = F.conv2d(x, self.haar, stride=2, groups=self.c)  # (B, 4C, H/2, W/2): LL, LH, HL, HH per channel
        # back to the input size (not scale_factor=2): an odd H or W (rect validation, e.g. 21) would be off by one
        s = F.interpolate(s, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return x + self.bn(self.fuse(s))


class C3k2Wave(nn.Module):
    """A pretrained C3k2 block followed by a Wave bypass."""

    def __init__(self, block):
        super().__init__()
        self.block = block
        self.wave = Wave(block.cv2.conv.out_channels)

    def forward(self, x):
        return self.wave(self.block(x))


class Cw2PSA(nn.Module):
    """C2PSA layout: cv1 -> split -> (Wave branch, PSA branch) -> concat -> cv2. cv1, PSA block and cv2 keep their weights."""

    def __init__(self, c2psa):
        super().__init__()
        self.c = c2psa.c
        self.cv1, self.m, self.cv2 = c2psa.cv1, c2psa.m, c2psa.cv2
        self.wave = Wave(self.c)

    def forward(self, x):
        a, b = self.cv1(x).split((self.c, self.c), dim=1)
        return self.cv2(torch.cat((self.wave(a), self.m(b)), 1))


def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_wave(model, layers=WAVE_LAYERS):
    """Wrap the C3k2 at the given backbone layers with a Wave bypass. Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        _swap(model, i, C3k2Wave(old), "C3k2Wave")
    return model


def add_cw2psa(model, layer=CW2PSA_LAYER):
    """Turn the C2PSA at the given layer into Cw2PSA (all C2PSA weights kept). Returns the model."""
    old = model.model[layer]
    assert isinstance(old, C2PSA), f"layer {layer} is {type(old).__name__}, not C2PSA"
    _swap(model, layer, Cw2PSA(old), "Cw2PSA")
    return model


class WaveTrainerBase(DetectionTrainer):
    """Loads the pretrained weights first, then applies Wave and/or Cw2PSA."""

    WAVE = False
    CW2PSA = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        parts = []
        if self.WAVE:
            add_wave(model)
            parts.append(f"Wave after layers {WAVE_LAYERS}")
        if self.CW2PSA:
            add_cw2psa(model)
            parts.append(f"Cw2PSA at layer {CW2PSA_LAYER}")
        if RANK in {-1, 0}:
            print("Paper 34 backbone: " + ", ".join(parts))
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class WaveTrainer(WaveTrainerBase):
    WAVE = True


class Cw2PSATrainer(WaveTrainerBase):
    CW2PSA = True


class WaveCw2PSATrainer(WaveTrainerBase):
    WAVE = True
    CW2PSA = True


Writing wave_backbone.py


In [5]:
import importlib, wave_backbone
importlib.reload(wave_backbone)
from wave_backbone import add_wave, add_cw2psa, WaveTrainer, Cw2PSATrainer, WaveCw2PSATrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

def first(y):
    return y[0] if isinstance(y, (list, tuple)) else y

# structure check (untrained): base vs the three Paper 34 backbones
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
m_wave = add_wave(deepcopy(base)).eval()
m_cw = add_cw2psa(deepcopy(base)).eval()
m_both = add_cw2psa(add_wave(deepcopy(base))).eval()
with torch.no_grad():
    x = torch.rand(1, 3, 640, 640)
    yb = first(base(x))
    for tag, m in (("Wave", m_wave), ("Cw2PSA", m_cw), ("Wave + Cw2PSA", m_both)):
        print(f"warm start {tag:14s}: max |base - variant| = {(yb - first(m(x))).abs().max():.2e} (should be 0)")
    # forward check with an active Wave branch (BN scale set to 1 only for this check)
    for mod in m_both.modules():
        if isinstance(mod, wave_backbone.Wave):
            mod.bn.weight.data.fill_(1.0)
    _ = m_both(x)
    _ = m_both(torch.rand(1, 3, 640, 352))  # rect-style input (as in validation): P5 is 11 wide (odd)
    for mod in m_both.modules():
        if isinstance(mod, wave_backbone.Wave):
            mod.bn.weight.data.zero_()
print("layers 2, 4, 6, 8, 10 (both):", [m_both.model[i].type for i in (2, 4, 6, 8, 10)])
for tag, m in (("base", base), ("Wave", m_wave), ("Cw2PSA", m_cw), ("Wave + Cw2PSA", m_both)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:14s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

Overriding model.yaml nc=80 with nc=27
warm start Wave          : max |base - variant| = 0.00e+00 (should be 0)
warm start Cw2PSA        : max |base - variant| = 0.00e+00 (should be 0)
warm start Wave + Cw2PSA : max |base - variant| = 0.00e+00 (should be 0)
layers 2, 4, 6, 8, 10 (both): ['C3k2Wave', 'C3k2Wave', 'C3k2Wave', 'C3k2Wave', 'Cw2PSA']
base           Params 2.3801 M | GFLOPs 5.338
Wave           Params 2.7909 M | GFLOPs 7.457
Cw2PSA         Params 2.4459 M | GFLOPs 5.391
Wave + Cw2PSA  Params 2.8566 M | GFLOPs 7.510


## 4. Training

Only `bb_cw2psa` is trained in this notebook (about 3–4 h on a T4). The other two variants are switched off with `#`; they are trained in their own notebooks.

In [6]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# Kaggle stops a session after 12 h. If one epoch takes more than about 6.5 min (100 epochs would not fit),
# remove the # below: training then stops by itself after 11 h and keeps best.pt (report the epochs reached).
COMMON["time"] = 11.0

# (row label in the table, run name, trainer)
VARIANTS = [
    # ("Wave Backbone (Paper 34)",          "bb_wave",        WaveTrainer),
    ("Cw2PSA Backbone (Paper 34)",        "bb_cw2psa",      Cw2PSATrainer),
    # ("Wave + Cw2PSA Backbone (Paper 34)", "bb_wave_cw2psa", WaveCw2PSATrainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run, trainer in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== Cw2PSA Backbone (Paper 34) (bb_cw2psa) ==========
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=N

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/280      3.21G      1.356      2.176    0.01845         14        640: 100% ━━━━━━━━━━━━ 482/482 4.6it/s 1:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.3it/s 2.5s
                   all        550        557      0.777      0.808       0.86      0.494

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/318      3.22G      1.401      1.473    0.01903         12        640: 100% ━━━━━━━━━━━━ 482/482 4.7it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.2it/s 2.5s
                   all        550        557      0.833      0.784      0.882      0.494

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/335      3.22G      1.429      1.218    0.01947          7        640: 100% ━━━━━━━━━━━━ 482/482 4.8it/s 1:41
                 Class     Images  Instances      Box(P 

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [7]:
import wave_backbone  # must be imported before loading a Paper 34 checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 124 layers, 2,445,893 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 885.7±305.9 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.3Kit/s 0.4s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 12.9it/s 2.7s
                   all        550        552      0.972      0.953      0.962      0.594
Speed: 0.5ms preprocess, 2.6ms inference, 0.0ms loss, 0.1ms postprocess per image


,Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,Cw2PSA Backbone (Paper 34),2.45,5.39,0.972,0.953,0.962,0.962


## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper34_bb_cw2psa_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_wave_ncnn_model/`, `bb_cw2psa_ncnn_model/`, `bb_wave_cw2psa_ncnn_model/` (NCNN, imgsz 640) | Raspberry Pi 5 |
| `bb_wave.pt`, `bb_cw2psa.pt`, `bb_wave_cw2psa.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `wave_backbone.py` | Jetson (needed to load the Paper 34 `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper34_bb_cw2psa_edge_bench.zip paper34_bb_cw2psa_edge_bench`.

In [8]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper34_bb_cw2psa_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("wave_backbone.py", EDGE / "wave_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    try:
        p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
        shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
        print(f"✅ {run:16s} -> {run}.pt, {run}_ncnn_model/")
    except Exception as e:
        print(f"❌ {run:16s} NCNN export failed: {e}  (the .pt for Jetson is still packaged)")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import wave_backbone  # needed to load the Paper 34 checkpoints
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper34_bb_cw2psa_edge_bench"), "zip", EDGE)
print("\nDownload paper34_bb_cw2psa_edge_bench.zip from the Output tab.")

bench_images: 50 real test images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 148 layers, 2,570,626 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from '/kaggle/working/runs/bb_cw2psa_s0/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (5.3 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 233ms
 Downloaded ncnn
Prepared 1 package in 231ms
Installed 1 package in 3ms
 + ncnn==1.0.20261020

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: 

pnnxparam = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.b

NCNN: export success ✅ 7.2s, saved as '/kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model' (9.5 MB)

Export complete (7.7s)
Results saved to /kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/bb_cw2psa_s0/weights/best_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
✅ bb_cw2psa        -> bb_cw2psa.pt, bb_cw2psa_ncnn_model/

Download paper34_bb_cw2psa_edge_bench.zip from the Output tab.


### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the three Paper 34 backbone models
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [9]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_cw2psa": None}
FPS_JETSON = {"bb_cw2psa": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Standard YOLO26n Backbone,2.380000,5.340000,0.971000,0.937000,0.953000,0.958000,13.380000,--
Cw2PSA Backbone (Paper 34),2.450000,5.390000,0.972000,0.953000,0.962000,0.962000,--,--


## 8. Zip all results

Run this last. It creates **`paper34_bb_cw2psa_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper34_bb_cw2psa_edge_bench/` | NCNN + `.pt` models, 50 bench images, `wave_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `bb_wave_best.pt`, `bb_cw2psa_best.pt`, `bb_wave_cw2psa_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [10]:
import zipfile

ALL_ZIP = WORK / "paper34_bb_cw2psa_all_results.zip"
folders = [WORK / "paper34_bb_cw2psa_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper34_bb_cw2psa_edge_bench/  59 files
runs/                          29 files
weights/bb_cw2psa_best.pt

Download paper34_bb_cw2psa_all_results.zip (46.9 MB) from the Output tab.
